# TP4 : Implémentation d'un système RAG (Retrieval-Augmented Generation)

## Présentation du TP

Ce travaux pratiques (TP) est dédié à la mise en place et à l'évaluation d'une architecture **RAG** (*Retrieval-Augmented Generation*). L'objectif principal est de pallier les limites de connaissances d'un grand modèle de langage (LLM) en lui permettant d'extraire et de s'appuyer sur des informations issues de documents externes spécifiques avant de générer une réponse.

Au cours de ce TP, nous allons aborder les étapes clés suivantes :
* **Le Document Loading & Splitting** : Chargement et découpage des documents en blocs (*chunks*).
* **L'Indexation & Vectorisation** : Transformation des textes en vecteurs et stockage dans une base de données vectorielle.
* **Le Retrieval (Recherche)** : Extraction des segments de texte les plus pertinents vis-à-vis d'une requête utilisateur.
* **La Génération & Évaluation** : Injection du contexte dans le prompt du LLM et analyse de la fidélité de ses réponses.


# 0. Preparation de l'environement 

Dans le terminal `uv init` , ` uv venv `, ` Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process ` et ` .venv\Scripts\activate ` .  ` uv add ipykernel ` et  ` uv add langchain ` puis ` uv add langchain-openai ` et  `uv add langchain-text-splitters ` puis  `uv add langchain-community ` finalement pour importer la clee api open ai ` uv add python-dotenv  `


In [ ]:
import json
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader 
from langchain_community.vectorstores import Chroma
from dotenv.ipython import load_dotenv
import os

C:\Users\pop\AppData\Local\Temp\ipykernel_7444\4814559.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader


##  LLM

In [2]:
load_dotenv(override=True)

True

In [5]:
from langchain_groq import ChatGroq
from dotenv.ipython import load_dotenv

In [6]:
from langchain_openai import ChatOpenAI
llm=ChatGroq(model="openai/gpt-oss-120b",temperature=0)

# 1. Loading the pdf, Chunking

Je vais créer un dossier que je vais appeler `PDFs` et à l'intérieur duquel je vais coller ce fichier PDF `nc346.pdf` (pour vous, choisissez votre CV par exemple, ou un fichier que vous connaissez déjà).


In [7]:
pdf_file="C:\\Users\\pop\\Downloads\\DAD\\Ahlame_Laouad_ATS_Resume (1).pdf"

In [8]:
pdf_loader=PyPDFLoader(pdf_file)

Et si vous voulez charger un dossier qui contient plusieurs fichiers, vous utilisez `PyPDFDirectoryLoader(path="./pdfs")`.

On va utiliser le découpage (*split*). Je rappelle donc que l'on utilise `RecursiveCharacterTextSplitter` avec l'encodeur `tiktoken`. Voici le modèle Tiktoken utilisé : la taille de chaque bloc (*chunk*) est de 300 tokens et le chevauchement (*overlap*) est de 20.

Maintenant, je vais charger le modèle `TextSplitter`. Notez qu'ici, il manque une dépendance. Pour effectuer le découpage, le système vous demande d'installer `pypdf`.

Faites donc : `uv add pypdf`.


In [9]:
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(encoding_name='o200k_base',chunk_size=300,chunk_overlap=20)

Dans le terminal de projet ajouter ` uv add pypdf `

La méthode `load_and_split` va charger le fichier, puis appliquer cette stratégie de découpage. Chaque bloc (*chunk*) aura une taille de 300 tokens avec un chevauchement (*overlap*) de 20. Si vous souhaitez vérifier le nombre total de blocs générés, vous pouvez utiliser `len(chunks)`.


In [10]:
chunks=pdf_loader.load_and_split(text_splitter)
len(chunks)


7

# 2.vectore Store - ChromaDB et Embedding

 *Importer le model d'embedding* 



In [21]:
#from langchain_google_genai import GoogleGenerativeAIEmbeddings
#embedding_model = GoogleGenerativeAIEmbeddings(
 #   model="text-embedding-004",
  #  google_api_key=os.getenv("GOOGLE_API_KEY")
from langchain_huggingface import HuggingFaceEmbeddings

# Modèle très léger (exécuté directement sans dépendre d'une API cloud payante)
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

c:\Users\pop\Downloads\DAD\python_uv_workspace\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\pop\Downloads\DAD\python_uv_workspace\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pop\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Pyth

Ajouter un dossier "store" et utiliser la commande ` uv add chromadb ` pour ajouter *chromadb*

 Je vais vectoriser les chunks et je les stock dans la base de données vectorielle ici **vectorstore**

In [22]:
#vectorstore=Chroma.from_documents(chunks,embedding_model,collection_name="nc346_v2",persist_directory="C:\\Users\\pop\\Downloads\\DAD\\store")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="nc346_v2",
    persist_directory=r"C:\Users\pop\Downloads\DAD\store"
)

# 3.  **Trouver les chunks liée à la questions (Retriever)**

In [23]:
#crée le Retriever 
retriever=vectorstore.as_retriever(search_type='similarity',search_Kwargs={'k',5})
# k=5. Donc je vais lui demander de me chercher les cinq chunks les plus proches à la question.


On va faire un test. J'ai la question suivante : 
> « Quelle est la capitalisation boursière du Maroc pour 2025 ? »

La méthode `retriever.invoke()` a recherché les blocs (*chunks*). Je vais maintenant les afficher.


In [25]:
#test
retrieved_chunks=retriever.invoke(" what is the last diploma ?")
print(retrieved_chunks)
print("nombre de chunks est :",len(retrieved_chunks))


[Document(metadata={'source': 'C:\\Users\\pop\\Downloads\\DAD\\Ahlame_Laouad_ATS_Resume (1).pdf', 'producer': 'www.ilovepdf.com', 'page': 0, 'total_pages': 3, 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-10-19T17:16:22+00:00', 'moddate': '2026-10-19T17:16:23+00:00', 'author': 'Un-named', 'page_label': '1'}, page_content="• Active research in multilingual sentiment analysis and LLM-based agentic pipelines. \nBachelor's Degree — Mathematical & Computer Sciences  |  Ibn Zohr Faculty of Sciences, AgadirSep 2024 – Jun 2025 \n• Specialization: Software Engineering. Graduated with distinction in software design and algorithm engineering. \nDEUG — Mathematical & Computer Sciences  |  Ibn Zohr Faculty of Sciences, Agadir Sep 2021 – May 2024 \n• Foundation in mathematics, algorithms, data structures, and object-oriented programming. \nInternational Baccalaureate (French Option)  |  Al Khawarizmi High School, Guelmim 2020 – 2021 \n \nPROFESSIONAL EXPERIENCE \nMachine Learning Engineer

Maintenant, pour la partie RAG, on va créer notre prompt système (*system prompt*).


# 4.  *Prompt Design*

## 4.1 Prompt template

In [26]:
prompt_template = """You are an expert AI recruiter and assistant.
Answer the user question accurately based ONLY on the provided candidate's ATS resume context.

Guidelines:
- Rely strictly on the facts, skills, education, and experiences provided in the <context>.
- Do not invent, extrapolate, or assume information that is not explicitly stated.
- Answer in the same language as the user question (if asked in French, answer in French; if in English, answer in English).
- If the answer is not found in the context, answer strictly: "Je ne sais pas !"

<context>
{context}
</context>

<question>
{question}
</question>
"""

### Note sur le contexte
Vous pouvez ajouter cette précision si vous savez que votre document concerne la Direction des études et des prévisions financières (DEPF). Vous indiquez alors le contexte suivant : *« Le contexte concerne la Note de conjoncture de la Direction des études et des prévisions financières de décembre 2025. »* C'est une phrase utile. Si vous préférez que le prompt reste général, quel que soit le fichier PDF, vous pouvez simplement retirer cette partie.

Le contexte est délimité par la balise `<context>`, et la question par la balise `<question>`.

### Gestion des hallucinations
Pour éviter les hallucinations, spécifiez dans le prompt : *« Si la réponse ne figure pas dans ce contexte, répondez par : "Je ne sais pas !" »*. Cela permet de régler efficacement le problème des hallucinations.


## 4.2 *Retrieving the Relevant Documents*

### Passage à la partie *retrieving*

Voici ma démarche : je récupère les documents, puis je les **convertis**. Pour chaque document (`relevant_document_chunks`), je sélectionne uniquement le contenu de la page (`d.page_content`), ce qui me génère une liste de textes. Enfin, nous allons concaténer cette liste de *chunks* de texte en utilisant le séparateur `". "`.


In [32]:
user_input="est ce qu'elle a de l'expérience professionnelle ? "

relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)


context_for_query

"AHLAME LAOUAD \nAI Engineer & Machine Learning Specialist \nlaouad.ahlame@gmail.com  |  +212 614 046 188  |  Beni Mellal, Morocco \nlinkedin.com/in/ahlame-laouad-14315a335  |  github.com/Missesqueensy \n \nPROFESSIONAL SUMMARY \nResults-driven AI Engineer and Master's candidate in Artificial Intelligence and Digital Computing (FST Béni Mellal, USMS) with \nhands-on expertise in deep learning, natural language processing, agentic AI systems, and full-stack ML pipelines. Proven ability to \ndesign and deploy end-to-end AI solutions — from data collection and model fine-tuning to production APIs and multi-agent \norchestration. Experienced with transformer-based NLP (BERT, DarijaBERT), LangGraph, LangChain, Groq LLMs, FastAPI, and Docker. \nFluent in Arabic, French, and English, with a strong track record of bridging research and implementation in multilingual AI contexts. \nActively targeting Junior AI Engineer, ML Engineer, NLP Engineer, or Data Scientist roles in international tech en

In [30]:
len(relevant_document_chunks)

4

## 4.3 Le prompt

Maintenant, on va injecter dans le prompt le contexte et la question en utilisant template format. Vous voyez donc pour template On a injecté le contexte et la question. Et puis donc le si vous voulez afficher le prompt. Qu'est-ce qu'il contient ? Il contient:

 1. le système message:  ( Answer the following question ............  Je ne sais pas! `` <context> ``).
 2. le contexte:  entre ` <context> ` et `` </context> `` .
 3. question:  entre ` <question> ` et ` </question> ` .




In [31]:
prompt=prompt_template.format(context=context_for_query,question=user_input)
print(prompt)

You are an expert AI recruiter and assistant.
Answer the user question accurately based ONLY on the provided candidate's ATS resume context.

Guidelines:
- Rely strictly on the facts, skills, education, and experiences provided in the <context>.
- Do not invent, extrapolate, or assume information that is not explicitly stated.
- Answer in the same language as the user question (if asked in French, answer in French; if in English, answer in English).
- If the answer is not found in the context, answer strictly: "Je ne sais pas !"

<context>
sources in real time. 
• Automated multi-channel delivery of results via SMTP email and Telegram audio messages using n8n orchestration and LLM-
generated summaries. 
Multi-Agent 'Petit Bac' Game  —  Java · JADE (Java Agent Development Framework) 
• Implemented autonomous JADE agents to manage game logic, real-time player interactions, and score validation across 
multiple concurrent players. 
• Designed inter-agent communication and synchronization 

Donc, ça c'est le prompt.

# 5. *Récupérer la réponse de la part de llm*

 Maintenant on va utiliser le prompt. On va poser cette question à LLM.

In [33]:

resp=llm.invoke(prompt)
from IPython.display import Markdown
print(display(Markdown(resp.content)))

You are currently a student at **FST Béni Mellal (USMS)**, pursuing your M.S. in Artificial Intelligence & Digital Computing.

None


## 6. Regroupement dans une application RAG


### Définition de la fonction RAG

Nous allons définir `RAG` comme notre fonction principale de RAG. Elle prendra en paramètres : `query`, `llm` et `prompt_template`.

**Comment fonctionne cette fonction RAG ?**
1. **Le Retriever** : On lui passe la question (`query`), il recherche les documents pertinents et les convertit en une liste de textes. Chaque texte représente le contenu d'une page (`page_content`).
2. **La Concaténation** : Nous allons concaténer cette liste de textes à l'aide d'un séparateur. Ce texte final devient notre **contexte**.
3. **La Génération** : On injecte ce contexte et la question dans le `prompt_template`. On appelle ensuite la méthode `invoke()` et on retourne le contenu généré.

Voici donc la fonction qui nous intéresse. Pour pouvoir la tester, il vous suffira de faire appel à `RAG()` à chaque fois que vous voudrez lui poser une question.



In [34]:
def RAG(query,llm=llm,prompt_template=prompt_template):
    context_docs=retriever.invoke(query)
    context_list=[d.page_content for d in context_docs]
    context_for_query=". ".join(context_list)   
    prompt=prompt_template.format(context=context_for_query, question=query)
    resp=llm.invoke(prompt)
    return resp.content

### Appel de RAG 

In [36]:
resp=RAG(" est ce qu'elle est une maman? ")
print(display(Markdown(resp)))

Je ne sais pas !

None


# 7. Évaluation de la réponse du LLM


## 7.1 Architecture

![Logo](./rag_juge.gif)

In [40]:
#question ou query 
user_input=" QU'ils sont les domaines d'intérêts de Ahlame Laouad ?"

Récupérer le context 

In [41]:
relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)

In [42]:
user_message_template="""
###Question
{question}
###Context
{context}
###Answer
{answer}
"""

In [43]:
# Réponse d'un RAG
answer=RAG(" Quels sont les domaines d'intérêts de Ahlame Laouad ? ? ")
print(display(Markdown(answer)))

Les domaines d’intérêt d’Ahlame Laouad, tels qu’ils ressortent de son CV, sont :

- **Intelligence artificielle** (IA) et **apprentissage automatique** (Machine Learning)  
- **Deep learning** et **modélisation de réseaux de neurones** (CNN, LSTM, GRU, NASNet‑Large)  
- **Traitement du langage naturel** (NLP) : tokenisation, classification de séquences, analyse de sentiment, NLP arabe et Darija, BERT/DarijaBERT, LangChain, LangGraph, LLMs (Llama‑3.3‑70B, Claude, Groq)  
- **Systèmes d’IA agentiques** et **orchestration multi‑agents** (JADE, n8n, routing agents)  
- **Pipelines ML end‑to‑end** : collecte de données, pré‑traitement, ingénierie des caractéristiques, gestion du déséquilibre de classes, fine‑tuning, déploiement d’API (FastAPI, Docker)  
- **MLOps / DevOps** : conteneurisation (Docker), gestion de versions (Git), entraînement GPU (Kaggle), bases de données NoSQL/SQL (MongoDB, MySQL, SQLite, PL/SQL)  
- **Vision par ordinateur** (mentionnée dans le cursus)  
- **Développement full‑stack** : Laravel, HTML5, CSS3, JavaScript, PHP, création d’interfaces web responsives et d’API RESTful  
- **Recherche académique** et **rédaction technique bilingue** (français/anglais)  

Ces domaines reflètent les compétences, les projets et la formation qu’il met en avant dans son parcours.

None


## 7.2- *Groundness*

Le prompt pour le juge 

In [44]:
groundness_rater_system_message="""
Vous êtes chargé d'évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d'IA pour générer la réponse, ainsi qu'une réponse générée par l'IA à la question.
Dans l'entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par L'IA commencera par ###Answer.
Critères d'évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1- La métrique n'est pas respectée du tout
2- La métrique n'est respectée que dans une mesure limitée
3- La métrique est respectée dans une bonne mesure
4- La métrique est respectée en grande partie
5- La métrique est entièrement respectée

Métrique :

La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions:

Écrivez d'abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d'évaluation et attribuer un score.
"""

In [45]:
# Concernant le juge, il est nécessaire d'employer un modèle plus puissant que celui mis en œuvre par le RAG. 
from langchain_openai import ChatOpenAI
groundness_checker=ChatGroq(model="openai/gpt-oss-120b",temperature=0)

In [46]:
def evaluate(system_message, user_message_template,question, model=groundness_checker):
    retrieved_chunks=retriever.invoke(question)
    context_list=[d.page_content for d in retrieved_chunks]
    context=". ".join(context_list)
    answer=RAG(question) 
    prompt=f"""
     { system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
     """
    juge_response=model.invoke(prompt)
    return juge_response.content



In [47]:
#évaluer
resp=evaluate(groundness_rater_system_message, user_message_template, user_input)
print(display(Markdown(resp)))

**Étapes pour évaluer la conformité à la métrique « la réponse doit être dérivée uniquement des informations présentées dans le contexte »**

1. **Identifier la question**  
   - Lire la partie commençant par `###Question` afin de comprendre ce qui est demandé.  
   - Ici : « Qu'ils sont les domaines d'intérêts de Ahlame Laouad ? »

2. **Analyser le contexte**  
   - Parcourir le texte sous `###Context` et relever toutes les informations qui décrivent les compétences, les projets, les expériences et les domaines d’expertise de Ahlame Laouad.  
   - Noter les termes clés (ex. : IA, Machine Learning, NLP, systèmes agentiques, MLOps, vision par ordinateur, développement full‑stack, marketing digital, etc.).

3. **Comparer le contenu de la réponse avec le contexte**  
   - Vérifier que chaque domaine mentionné dans la réponse apparaît explicitement ou implicitement dans le contexte.  
   - S’assurer qu’aucune information supplémentaire (non‑présente dans le contexte) n’a été ajoutée.

4. **Détecter les éventuelles extrapolations ou inventions**  
   - Rechercher des éléments qui ne sont pas soutenus par le texte du contexte (par exemple, des intérêts personnels, des hobbies, des domaines non cités).  
   - Si de tels éléments existent, la métrique n’est pas respectée.

5. **Évaluer le degré de conformité**  
   - Si toutes les informations de la réponse proviennent du contexte et aucune information externe n’est introduite, la métrique est **entièrement respectée** (score 5).  
   - Si la réponse omet des informations présentes dans le contexte ou ajoute du contenu non justifié, le score diminue proportionnellement (4 → 1).

---

**Application de ces étapes à l’exemple fourni**

1. **Question** : Quels sont les domaines d’intérêts de Ahlame Laouad ?  
2. **Informations pertinentes du contexte**  
   - *Intelligence artificielle & Machine Learning* (deep learning, transformers, BERT, LLMs).  
   - *Traitement du langage naturel* (NLP, sentiment analysis, DarijaBERT, tokenisation, classification).  
   - *Systèmes d’IA agentiques & multi‑agents* (JADE, LangGraph, LangChain, orchestration).  
   - *MLOps & déploiement* (Docker, FastAPI, API REST, pipelines).  
   - *Ingénierie des données* (pré‑traitement, feature engineering, bases MongoDB, MySQL, SQLite).  
   - *Vision par ordinateur* (NASNet‑Large).  
   - *Développement full‑stack* (Laravel, HTML5, CSS3, JavaScript, PHP).  
   - *Gestion de projets numériques & marketing digital* (leadership d’équipes à distance, campagnes en ligne).  

3. **Correspondance avec la réponse**  
   - La réponse liste exactement les huit domaines ci‑dessus, formulés avec les mêmes termes ou des synonymes proches.  
   - Aucun domaine supplémentaire n’est introduit et aucune information du contexte n’est omise de façon significative.

4. **Extrapolation ou invention**  
   - Aucun élément n’est présent dans la réponse qui ne figure pas déjà dans le contexte.  

5. **Évaluation**  
   - La réponse est entièrement dérivée du contexte, sans ajout externe.  

**Score attribué** : **5 – La métrique est entièrement respectée**.

None


### Échelle de notation pour la fidélité au contexte (Faithfulness)


| Score | Signification | Description |
| :--- | :--- | :--- |
| **5 / 5** | **Excellent (Parfait)** | La réponse utilise **uniquement** les informations du contexte. Aucune hallucination, aucune extrapolation. |
| **4 / 5** | **Bon** | La réponse est correcte et s'appuie sur le contexte, mais la formulation pourrait être légèrement plus précise. |
| **3 / 5** | **Moyen** | La réponse est partiellement vraie, mais elle omet un détail important du contexte ou manque de clarté. |
| **2 / 5** | **Médiocre** | La réponse contient des extrapolations ou des informations non vérifiables avec le contexte fourni. |
| **1 / 5** | **Mauvais (Hallucination)** | La réponse est totalement hors-sujet, fausse, ou invente des données qui ne sont pas du tout dans le contexte. |
